# Beyond - Midtraining: Continued Pretraining and Data Mixtures

This notebook forks one TinyLLM base checkpoint into equal-token Python-only and Python-plus-replay branches, then measures target adaptation and retention on three frozen domains.

1. Read the lesson page (`docs/beyond/midtraining.md`).
2. Open this notebook with `./notebook.sh midtraining`.
3. Answer the `Question:` / `Answer:` cells below.
4. When you're ready, ask a coding agent to grade your notebook.

Partial work is fine. Blank `Answer: ""` strings are skipped, not counted wrong. If you'd like a hint instead of a grade, write the request inline in the answer string and the agent will tutor first.

In [ ]:
import ast
from pathlib import Path

import torch

import g2c
from g2c.artifacts import load_model_artifact_with_tokenizer, save_model_artifact
from g2c.midtraining import TokenMixture, evaluate_domain_losses
from g2c.notebook_extras.midtraining import (
    load_g2c_source_ids, plot_domain_loss_deltas, show_domain_loss_table,
)
from g2c.notebook_extras.pretraining import train_with_progress
from g2c.notebook_extras.sampling import sample_model_text

repo_root = Path(g2c.__file__).resolve().parents[1]
device = "mps" if torch.backends.mps.is_available() else "cpu"
WORK = repo_root / "data" / "work" / "beyond-midtraining"
WORK.mkdir(parents=True, exist_ok=True)
print("repo root:", repo_root)
print("training device:", device)

## Exercise 1 — Establish the base and frozen domains

Load the same `TinyLLM-30M-base` checkpoint both branches will fork. The bounded source loader reads only small slices of G2C Corpus v1: Python for adaptation, non-code FineWeb-Edu/Cosmopedia for general retention, and TinyStories as a visibly different retention domain. Validation shards are fixed before either branch trains.

In [ ]:
BASE_ARTIFACT = "TinyLLM-30M-base"
CORPUS = "g2c-corpus-small"
VOCAB_SIZE = 8192
TRAIN_BYTES_PER_DOMAIN = 8_000_000
VAL_BYTES_PER_DOMAIN = 1_000_000

base_artifact = load_model_artifact_with_tokenizer(
    BASE_ARTIFACT, repo_root=repo_root, device="cpu"
)
tokenizer = base_artifact.tokenizer

def source_ids(names, split, byte_count):
    return load_g2c_source_ids(
        CORPUS, tuple(names), split=split, byte_count=byte_count,
        tokenizer=tokenizer, vocab_size=VOCAB_SIZE, repo_root=repo_root,
    )

python_train = source_ids(["codesearchnet-python"], "train", TRAIN_BYTES_PER_DOMAIN)
general_train = source_ids(
    ["fineweb-edu-dedup", "cosmopedia-v2", "tinystories"],
    "train", TRAIN_BYTES_PER_DOMAIN,
)
validation_domains = {
    "python": source_ids(["codesearchnet-python"], "val", VAL_BYTES_PER_DOMAIN),
    "general": source_ids(
        ["fineweb-edu-dedup", "cosmopedia-v2"], "val", VAL_BYTES_PER_DOMAIN
    ),
    "stories": source_ids(["tinystories"], "val", VAL_BYTES_PER_DOMAIN),
}
print("loaded:", base_artifact.name)
print("parameters:", f"{sum(p.numel() for p in base_artifact.model.parameters()):,}")
print("training tokens available:", len(python_train), len(general_train))
print("validation tokens:", {k: len(v) for k, v in validation_domains.items()})

In [ ]:
EVAL_CONFIG = {
    "batch_size": 4, "context_length": 256, "eval_iters": 10,
    "device": device, "seed": 41,
}
base_losses = evaluate_domain_losses(
    base_artifact.model, validation_domains, **EVAL_CONFIG
)
print("base losses:", {k: round(v, 4) for k, v in base_losses.items()})

In [ ]:
"Question: What makes this run midtraining rather than SFT? Identify the data shape, which tokens receive causal loss, and why updating every model weight does not by itself distinguish the two stages."
"Answer: "

## Exercise 2 — Build and inspect the mixture

`TokenMixture` chooses a source for each batch row and only then samples a contiguous window inside it. Run enough small batches to see configured weights become realized frequencies.

In [ ]:
python_only_probe = TokenMixture(
    {"python": python_train, "general": general_train},
    {"python": 1.0, "general": 0.0},
)
replay_probe = TokenMixture(
    {"python": python_train, "general": general_train},
    {"python": 0.8, "general": 0.2},
)
generator = torch.Generator().manual_seed(17)
for _ in range(100):
    replay_probe.get_lm_batch(20, 32, generator=generator)
print(python_only_probe)
print(replay_probe)
print("realized over 2,000 examples:", replay_probe.observed_fractions)

In [ ]:
"Question: Why does the mixture choose a source before sampling a window, and why is equal TOTAL token count the fairness control here? Explain what would change if the branches instead received equal Python-token exposure."
"Answer: "

## Exercise 3 — Train equal-token branches

Both branches reload the identical base artifact and consume `300 × 4 × 256 = 307,200` tokens. The lower continuation learning rate briefly rewarms, then decays. Rolling checkpoints make interruption safe; rerunning a completed cell resumes rather than restarting.

In [ ]:
SEED = 17
TRAIN_CONFIG = {
    "batch_size": 4,
    "context_length": 256,
    "max_steps": 300,
    "max_lr": 2e-5,
    "min_lr": 2e-6,
    "warmup_steps": 20,
    "weight_decay": 0.1,
    "grad_clip": 1.0,
    "eval_every": 100,
    "eval_iters": 5,
    "log_every": 10,
    "device": device,
    "optimizer": "adamw",
}
TOKEN_BUDGET = (TRAIN_CONFIG["max_steps"] * TRAIN_CONFIG["batch_size"]
                * TRAIN_CONFIG["context_length"])
print(f"equal token budget per branch: {TOKEN_BUDGET:,}")

In [ ]:
python_artifact = load_model_artifact_with_tokenizer(
    BASE_ARTIFACT, repo_root=repo_root, device="cpu"
)
python_model = python_artifact.model
python_mixture = TokenMixture(
    {"python": python_train, "general": general_train},
    {"python": 1.0, "general": 0.0},
)
python_history = train_with_progress(
    "Python-only midtraining", python_model, python_mixture,
    validation_domains["python"], seed=SEED,
    checkpoint_path=WORK / "python-only.ckpt", checkpoint_every=100,
    checkpoint_extra={
        "model_config": base_artifact.manifest["model_config"],
        "vocab_size": VOCAB_SIZE, "tokenizer_artifact": "G2CTokenizer",
        "base_artifact": BASE_ARTIFACT, "mixture": {"python": 1.0},
    },
    **TRAIN_CONFIG,
)
python_dir = save_model_artifact(
    "TinyLLM-30M-mid-python", model=python_model,
    model_config=base_artifact.manifest["model_config"],
    training_config={**TRAIN_CONFIG, "token_budget": TOKEN_BUDGET,
                     "mixture": {"python": 1.0, "general": 0.0}},
    tokenizer_artifact_name="G2CTokenizer",
    source="CodeSearchNet Python from G2C Corpus v1 small",
    history=python_history, seed=SEED, module="beyond-midtraining",
    notes="Equal-token Python-only branch for adaptation/retention comparison.",
    base_artifact_name=BASE_ARTIFACT, repo_root=repo_root,
)
print("saved", python_dir.relative_to(repo_root))

In [ ]:
replay_artifact = load_model_artifact_with_tokenizer(
    BASE_ARTIFACT, repo_root=repo_root, device="cpu"
)
replay_model = replay_artifact.model
replay_mixture = TokenMixture(
    {"python": python_train, "general": general_train},
    {"python": 0.8, "general": 0.2},
)
replay_val = TokenMixture(
    {"python": validation_domains["python"],
     "general": validation_domains["general"]},
    {"python": 0.8, "general": 0.2},
)
replay_history = train_with_progress(
    "Python + replay midtraining", replay_model, replay_mixture, replay_val,
    seed=SEED, checkpoint_path=WORK / "python-replay.ckpt",
    checkpoint_every=100, checkpoint_extra={
        "model_config": base_artifact.manifest["model_config"],
        "vocab_size": VOCAB_SIZE, "tokenizer_artifact": "G2CTokenizer",
        "base_artifact": BASE_ARTIFACT,
        "mixture": {"python": 0.8, "general": 0.2},
    },
    **TRAIN_CONFIG,
)
replay_dir = save_model_artifact(
    "TinyLLM-30M-mid-python-replay", model=replay_model,
    model_config=base_artifact.manifest["model_config"],
    training_config={**TRAIN_CONFIG, "token_budget": TOKEN_BUDGET,
                     "mixture": {"python": 0.8, "general": 0.2}},
    tokenizer_artifact_name="G2CTokenizer",
    source="80% CodeSearchNet Python + 20% non-code G2C replay",
    history=replay_history, seed=SEED, module="beyond-midtraining",
    notes="Equal-token replay branch for adaptation/retention comparison.",
    base_artifact_name=BASE_ARTIFACT, repo_root=repo_root,
)
print("saved", replay_dir.relative_to(repo_root))
print("realized training examples in this kernel segment:",
      replay_mixture.example_counts, replay_mixture.observed_fractions)

In [ ]:
"Question: Verify that both runs forked the same base and consumed 307,200 tokens. Why use a fresh optimizer with a short rewarm and a peak learning rate below the original pretraining peak? What confound would appear if replay started from the completed Python-only branch?"
"Answer: "

## Exercise 4 — Measure adaptation versus retention

Every checkpoint sees the same seeded validation windows. Parenthesized values are loss changes from the base: negative is improvement, positive is degradation.

In [ ]:
comparison_models = {
    "base": base_artifact.model,
    "python-only": python_model,
    "80/20 replay": replay_model,
}
domain_results = {
    name: evaluate_domain_losses(model, validation_domains, **EVAL_CONFIG)
    for name, model in comparison_models.items()
}
show_domain_loss_table(domain_results)
plot_domain_loss_deltas(domain_results)

In [ ]:
"Question: Interpret all three domain columns. How much target adaptation did each branch gain, where did Python-only forget, and what target gain did replay sacrifice to improve retention? If your pattern differs from the reference expectation, report the result rather than forcing the story."
"Answer: "

## Exercise 5 — Inspect behavior without promoting anecdotes to metrics

Use identical prompts and decoding settings for every checkpoint. The syntax probe is explicitly experimental: at 30M scale it may be sparse, noisy, or uniformly zero. Keep the raw outputs either way.

In [ ]:
PYTHON_PROMPT = "def fibonacci(n):\n    "
PROSE_PROMPT = "The most important lesson was"
for name, model in comparison_models.items():
    print(f"\n{'=' * 24} {name} / Python {'=' * 24}")
    print(sample_model_text(
        model, tokenizer, PYTHON_PROMPT, max_new_tokens=100,
        temperature=0.5, top_k=20, seed=7,
    ))
    print(f"\n{name} / prose")
    print(sample_model_text(
        model, tokenizer, PROSE_PROMPT, max_new_tokens=100,
        temperature=0.5, top_k=20, seed=7,
    ))

print("\nExperimental Python syntax probe (3 fixed seeds):")
for name, model in comparison_models.items():
    valid = 0
    for seed in range(3):
        text = sample_model_text(
            model, tokenizer, PYTHON_PROMPT, max_new_tokens=60,
            temperature=0.5, top_k=20, seed=seed,
        )
        try:
            ast.parse(text)
        except SyntaxError:
            pass
        else:
            valid += 1
    print(f"{name:<14} {valid}/3 parse as Python")

In [ ]:
"Question: What changed in the fixed Python and prose samples, and does the syntax-validity probe have enough variation to be informative? Explain why held-out domain loss remains stronger evidence than choosing the most attractive generation."
"Answer: "

## Exercise 6 — Optional replay-ratio sweep

Repeat shorter 75-step branches at 100/0, 90/10, 80/20, and 50/50. Hold total tokens, schedule, seed, and starting checkpoint fixed. Plot Python loss against general loss: the result is a frontier, not a universally best ratio. This optional block retrains four short-lived in-memory models and saves no artifacts.

In [ ]:
SWEEP_STEPS = 75
sweep_results = {}
for python_weight in (1.0, 0.9, 0.8, 0.5):
    general_weight = 1.0 - python_weight
    sweep_artifact = load_model_artifact_with_tokenizer(
        BASE_ARTIFACT, repo_root=repo_root, device="cpu"
    )
    sweep_mix = TokenMixture(
        {"python": python_train, "general": general_train},
        {"python": python_weight, "general": general_weight},
    )
    sweep_config = {**TRAIN_CONFIG, "max_steps": SWEEP_STEPS,
                    "warmup_steps": 10, "eval_every": SWEEP_STEPS + 1}
    train_with_progress(
        f"sweep {python_weight:.0%}/{general_weight:.0%}",
        sweep_artifact.model, sweep_mix, None, seed=SEED,
        checkpoint_path=None, **sweep_config,
    )
    sweep_results[f"{python_weight:.0%}/{general_weight:.0%}"] = (
        evaluate_domain_losses(
            sweep_artifact.model,
            {"python": validation_domains["python"],
             "general": validation_domains["general"]},
            **EVAL_CONFIG,
        )
    )
print(sweep_results)

In [ ]:
"Question (optional): Describe the replay-ratio frontier under equal total tokens. Which ratio would you choose if Python adaptation and general retention had equal value, and what additional evidence would you want before treating that choice as a recipe?"
"Answer: "

When complete, ask a coding agent to grade your notebook. Partial work is fine: the agent should grade answered questions and implemented sections, then skip blank prompts.